# Regression and Time Series Modelling of Students' Performance Across Semester

**Author:** Ehime Kelvin Ehinomen — Mountain Top University, Jan 2026  
**Data Engineering:** Yaba-Shiaka, Shemaiah Wambebe  
**Dataset:** `academic_performance_enriched.csv` — 3,046 records, 17 variables  

---

This notebook implements the full analysis pipeline:
1. Setup & data loading
2. Descriptive statistics and EDA
3. OLS regression with full diagnostics
4. Time series — CGPA trajectory across Levels 100→400
5. Limitations (L1–L6) — required disclosure

> **Reproducibility:** All synthesis is seeded (seed=42). Re-running `enrich.py` with the same seed produces an identical enriched CSV.

---
## Section 1 — Setup & Data Loading

In [ ]:
import os
import subprocess
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import statsmodels.api as sm
from scipy import stats

# ── Plotting style ────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (10, 5)

print(f"Python  {sys.version.split()[0]}")
print(f"pandas  {pd.__version__}")
print(f"numpy   {np.__version__}")
print(f"statsmodels loaded")

In [ ]:
CSV_PATH = "academic_performance_enriched.csv"
RAW_PATH = "academic_performance_dataset_V2.csv"

# Regenerate enriched CSV if missing
if not os.path.exists(CSV_PATH):
    print(f"{CSV_PATH} not found — regenerating via enrich.py ...")
    result = subprocess.run(
        [sys.executable, "enrich.py",
         "--input", RAW_PATH,
         "--output", CSV_PATH,
         "--seed", "42"],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        raise RuntimeError(f"enrich.py failed:\n{result.stderr}")
    print(result.stdout.strip())

# ── Load ──────────────────────────────────────────────────────────────────────
df_raw = pd.read_csv(CSV_PATH)
print(f"Loaded  : {len(df_raw):,} rows × {df_raw.shape[1]} columns")

# ── De-duplicate on ID No ─────────────────────────────────────────────────────
# 70 ID numbers appear on rows with conflicting programme/gender/YoG attributes.
# These are data-entry errors in the source (see memory.md §1, Limitation L5).
# We keep the first occurrence and drop the collision row before any analysis.
n_before = len(df_raw)
df = df_raw.drop_duplicates(subset="ID No", keep="first").reset_index(drop=True)
n_dropped = n_before - len(df)
print(f"De-duped: {len(df):,} rows  ({n_dropped} duplicate ID rows dropped)")

In [ ]:
# ── Schema & dtypes ───────────────────────────────────────────────────────────
print(df.dtypes.to_string())
print(f"\nNull counts:\n{df.isnull().sum().to_string()}")

In [ ]:
# ── Descriptive statistics ────────────────────────────────────────────────────
numeric_cols = [
    "CGPA", "Previous_GPA",
    "CGPA100", "CGPA200", "CGPA300", "CGPA400", "SGPA",
    "Attendance_Rate", "Study_Hours_Per_Week", "Course_Load"
]
df[numeric_cols].describe().T.round(3)

---
## Section 2 — Descriptive Statistics & EDA

In [ ]:
# ── 2.1 CGPA distribution ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram + KDE
sns.histplot(df["CGPA"], bins=40, kde=True, ax=axes[0], color="steelblue")
axes[0].set_title("CGPA Distribution (all students, post-dedup)")
axes[0].set_xlabel("CGPA")
axes[0].axvline(df["CGPA"].mean(), color="firebrick", linestyle="--",
                label=f"Mean = {df['CGPA'].mean():.2f}")
axes[0].legend()

# Box by Gender
sns.boxplot(data=df, x="Gender", y="CGPA", palette="Set2", ax=axes[1])
axes[1].set_title("CGPA by Gender")

plt.tight_layout()
plt.show()

In [ ]:
# ── 2.2 Programme breakdown ───────────────────────────────────────────────────
prog_order = (
    df.groupby("Prog Code")["CGPA"]
    .median()
    .sort_values(ascending=False)
    .index.tolist()
)

fig, ax = plt.subplots(figsize=(14, 5))
sns.boxplot(
    data=df, x="Prog Code", y="CGPA",
    order=prog_order, palette="tab20", ax=ax
)
ax.set_title("CGPA by Programme (sorted by median)")
ax.set_xlabel("Programme Code")
ax.set_ylabel("CGPA")
plt.tight_layout()
plt.show()

# Student counts per programme
df["Prog Code"].value_counts().rename("Count").to_frame()

In [ ]:
# ── 2.3 Synthesized variable distributions ───────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, col, color in zip(
    axes,
    ["Attendance_Rate", "Study_Hours_Per_Week", "Course_Load"],
    ["teal", "darkorange", "mediumpurple"]
):
    sns.histplot(df[col], bins=30, kde=True, ax=ax, color=color)
    ax.set_title(col.replace("_", " "))
    ax.axvline(df[col].mean(), color="black", linestyle="--",
               label=f"Mean = {df[col].mean():.1f}")
    ax.legend(fontsize=9)

plt.suptitle("Synthesized Variable Distributions", y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── 2.4 Correlation heatmap ───────────────────────────────────────────────────
corr_cols = [
    "CGPA", "Previous_GPA",
    "CGPA100", "CGPA200", "CGPA300", "CGPA400",
    "Attendance_Rate", "Study_Hours_Per_Week", "Course_Load"
]
corr = df[corr_cols].corr().round(3)

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)  # show lower triangle
sns.heatmap(
    corr, annot=True, fmt=".2f",
    cmap="RdBu_r", vmin=-1, vmax=1,
    linewidths=0.5, ax=ax
)
ax.set_title("Pearson Correlation Matrix", fontsize=13)
plt.tight_layout()
plt.show()

> **Key observation:** Primary regression uses **CGPA400** as the dependent variable; `Previous_GPA` is arithmetically prior (mean of CGPA100–300). Overall `CGPA` still overlaps `Previous_GPA` (legacy r ≈ 0.976) — see **Limitation L2** in Section 5.

---
## Section 3 — OLS Regression

**Model:** `CGPA400 ~ Previous_GPA + Trajectory_Slope_Prior + Attendance_Rate + Study_Hours_Per_Week + Course_Load + Genotype_AS + Genotype_SS + Genotype_SS×Attendance`

In [ ]:
# ── 3.1 Fit OLS ───────────────────────────────────────────────────────────────
from prediction_common import DESIGN_COLUMNS, fit_reference_ols

y = df["CGPA400"]
model = fit_reference_ols(df, target="CGPA400")
PREDICTORS = DESIGN_COLUMNS  # for cells below that reference coefficient names
print(model.summary())

In [ ]:
# ── 3.2 Coefficient table ─────────────────────────────────────────────────────
ci = model.conf_int()
coef_df = pd.DataFrame({
    "Coefficient": model.params.round(4),
    "Std Error":   model.bse.round(4),
    "t-statistic": model.tvalues.round(3),
    "p-value":     model.pvalues.round(4),
    "CI Lower (95%)": ci[0].round(4),
    "CI Upper (95%)": ci[1].round(4),
})

def highlight_pval(val):
    color = "background-color: #d4edda" if val < 0.05 else "background-color: #f8d7da"
    return color

coef_df.style.applymap(highlight_pval, subset=["p-value"])

In [ ]:
# ── 3.3 Coefficient plot ──────────────────────────────────────────────────────
# Exclude intercept from plot
plot_vars = DESIGN_COLUMNS
from prediction_common import COEF_LABELS
plot_labels = COEF_LABELS
coefs  = model.params[plot_vars]
errors = model.bse[plot_vars]

fig, ax = plt.subplots(figsize=(9, 4))
colors = ["#2196F3" if c > 0 else "#F44336" for c in coefs]
ax.barh(plot_labels, coefs, xerr=1.96 * errors, color=colors,
        capsize=5, alpha=0.85, edgecolor="white")
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("OLS Coefficients (± 1.96 SE)", fontsize=13)
ax.set_xlabel("Coefficient value")
plt.tight_layout()
plt.show()

In [ ]:
# ── 3.4 Residual diagnostics (2×2) ───────────────────────────────────────────
fitted = model.fittedvalues
resid  = model.resid
std_resid = resid / resid.std()

fig, axes = plt.subplots(2, 2, figsize=(12, 9))

# ── Residuals vs Fitted ───────────────────────────────────────────────────────
axes[0, 0].scatter(fitted, resid, alpha=0.3, s=10, color="steelblue")
axes[0, 0].axhline(0, color="firebrick", linewidth=1.2)
axes[0, 0].set_xlabel("Fitted values")
axes[0, 0].set_ylabel("Residuals")
axes[0, 0].set_title("Residuals vs Fitted")

# ── Q-Q plot ──────────────────────────────────────────────────────────────────
(osm, osr), (slope, intercept, _) = stats.probplot(resid.values, dist="norm")
axes[0, 1].scatter(osm, osr, alpha=0.3, s=10, color="steelblue")
axes[0, 1].plot(osm, slope * np.array(osm) + intercept,
                color="firebrick", linewidth=1.5)
axes[0, 1].set_xlabel("Theoretical quantiles")
axes[0, 1].set_ylabel("Sample quantiles")
axes[0, 1].set_title("Q-Q Plot of Residuals")

# ── Residual histogram + normal overlay ──────────────────────────────────────
sns.histplot(resid, bins=40, kde=False, stat="density",
             ax=axes[1, 0], color="steelblue", alpha=0.6)
x_norm = np.linspace(resid.min(), resid.max(), 200)
axes[1, 0].plot(x_norm,
                stats.norm.pdf(x_norm, resid.mean(), resid.std()),
                color="firebrick", linewidth=1.5, label="Normal curve")
axes[1, 0].set_title("Residual Distribution")
axes[1, 0].set_xlabel("Residual")
axes[1, 0].legend()

# ── Scale-Location ────────────────────────────────────────────────────────────
axes[1, 1].scatter(fitted, np.sqrt(np.abs(std_resid)),
                   alpha=0.3, s=10, color="steelblue")
axes[1, 1].set_xlabel("Fitted values")
axes[1, 1].set_ylabel(r"$\sqrt{|\mathrm{Std.\ Residuals}|}$")
axes[1, 1].set_title("Scale-Location")

plt.suptitle("OLS Residual Diagnostics", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

> **Q-Q plot caveat:** Residual shape still reflects synthesis (L1–L3) and the strong `Previous_GPA` → `CGPA400` association. With CGPA400 as DV, overlap with `Previous_GPA` is reduced versus overall `CGPA` (L2).

In [ ]:
# ── 3.5 Actual vs Predicted ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))

programmes = df["Prog Code"].unique()
palette = sns.color_palette("tab20", len(programmes))
prog_color = dict(zip(programmes, palette))

for prog in programmes:
    mask = df["Prog Code"] == prog
    ax.scatter(
        fitted[mask], y[mask],
        alpha=0.35, s=12, label=prog,
        color=prog_color[prog]
    )

lo, hi = min(y.min(), fitted.min()), max(y.max(), fitted.max())
ax.plot([lo, hi], [lo, hi], "k--", linewidth=1.2, label="Identity line")
ax.set_xlabel("Predicted CGPA400")
ax.set_ylabel("Actual CGPA400")
ax.set_title(f"Actual vs Predicted CGPA400  (Adj R² = {model.rsquared_adj:.3f})")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

> **Note on fit:** Adj R² on CGPA400 (≈ 0.65) is lower than models on overall CGPA (≈ 0.95) because `Previous_GPA` no longer shares a level component with the DV. Behavioural and genotype terms remain illustrative (L1, L3, L7).

---
## Section 3.6 — Out-of-sample evaluation and scoring

K-fold cross-validation reports **MAE**, **RMSE**, and **out-of-sample R²** stacked from held-out folds (CLI: `python predict.py`). Operational scoring for new rows refits the same OLS on the full deduped enriched dataset (`python score.py`). Limitations **L1–L3** still apply — see Section 5.

In [ ]:
from prediction_common import cross_val_ols_metrics, prepare_scoring_features, score_dataframe

cv = cross_val_ols_metrics(df, "CGPA400", k=5, seed=42)
print("K-fold (k=5) out-of-sample metrics, target=CGPA400:")
for key in ("mae", "rmse", "r2_oof", "baseline_mean_mae", "baseline_mean_rmse"):
    print(f"  {key}: {cv[key]:.6f}")

# Golden check: scorer matches in-sample fitted values on the training design matrix
X_design, _ = prepare_scoring_features(
    df[["Previous_GPA", "Attendance_Rate", "Study_Hours_Per_Week", "Course_Load", "Genotype"]]
)
pred_batch = score_dataframe(model, X_design).values
print(
    "max |batch prediction - fitted|:",
    np.max(np.abs(pred_batch - model.fittedvalues.values)),
)

# Toy hypothetical rows (illustrative only — not real students)
toy = pd.DataFrame({
    "CGPA100": [3.2, 2.8],
    "CGPA200": [3.4, 3.0],
    "CGPA300": [3.5, 3.1],
    "Attendance_Rate": [85.0, 72.0],
    "Study_Hours_Per_Week": [10.0, 7.5],
    "Course_Load": [12, 11],
    "Genotype": ["AA", "AS"],
})
Xt, _ = prepare_scoring_features(toy)
print("\nToy predicted CGPA400:\n", score_dataframe(model, Xt).to_string(index=False))

---
## Section 4 — Time Series: CGPA Trajectory Across Levels 100→400

> **Interpretation note:** `CGPA100`–`CGPA400` represent per-level academic performance (Level 100 = Year 1 through Level 400 = Year 4). Each student contributes one observation per level, so `CGPA100→400` form a **within-student progression sequence**. The group trajectories shown here are **means across students at each level**, not repeated measurements of the same cohort tracked over calendar time.

In [ ]:
# ── 4.1 Prepare long-format data ──────────────────────────────────────────────
LEVEL_COLS = ["CGPA100", "CGPA200", "CGPA300", "CGPA400"]
LEVEL_LABELS = ["Level 100", "Level 200", "Level 300", "Level 400"]

df_long = df[["ID No", "Prog Code", "YoG", "Gender"] + LEVEL_COLS].copy()
df_long = df_long.melt(
    id_vars=["ID No", "Prog Code", "YoG", "Gender"],
    value_vars=LEVEL_COLS,
    var_name="Level",
    value_name="GPA"
)
df_long["Level"] = df_long["Level"].map(
    dict(zip(LEVEL_COLS, LEVEL_LABELS))
)
print(f"Long-format shape: {df_long.shape}")
df_long.head(8)

In [ ]:
# ── 4.2 Mean trajectory by Programme ─────────────────────────────────────────
prog_traj = (
    df_long.groupby(["Prog Code", "Level"])["GPA"]
    .mean()
    .reset_index()
)
# Preserve level order
prog_traj["Level"] = pd.Categorical(
    prog_traj["Level"], categories=LEVEL_LABELS, ordered=True
)
prog_traj = prog_traj.sort_values(["Prog Code", "Level"])

# Only plot programmes with >= 20 students for readability
large_progs = df["Prog Code"].value_counts()
large_progs = large_progs[large_progs >= 20].index.tolist()
prog_traj_large = prog_traj[prog_traj["Prog Code"].isin(large_progs)]

fig, ax = plt.subplots(figsize=(12, 5))
palette_prog = sns.color_palette("tab20", len(large_progs))
for prog, color in zip(large_progs, palette_prog):
    d = prog_traj_large[prog_traj_large["Prog Code"] == prog]
    ax.plot(d["Level"], d["GPA"], marker="o", label=prog, color=color)

ax.set_title("Mean CGPA Trajectory by Programme (Level 100 → 400)")
ax.set_xlabel("Academic Level")
ax.set_ylabel("Mean GPA")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9, ncol=2)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))
plt.tight_layout()
plt.show()

In [ ]:
# ── 4.3 Mean trajectory by Year of Graduation ─────────────────────────────────
yog_traj = (
    df_long.groupby(["YoG", "Level"])["GPA"]
    .mean()
    .reset_index()
)
yog_traj["Level"] = pd.Categorical(
    yog_traj["Level"], categories=LEVEL_LABELS, ordered=True
)
yog_traj = yog_traj.sort_values(["YoG", "Level"])

fig, ax = plt.subplots(figsize=(10, 5))
palette_yog = sns.color_palette("Set1", yog_traj["YoG"].nunique())
for yog, color in zip(sorted(yog_traj["YoG"].unique()), palette_yog):
    d = yog_traj[yog_traj["YoG"] == yog]
    ax.plot(d["Level"], d["GPA"], marker="s", label=str(yog), color=color)

ax.set_title("Mean CGPA Trajectory by Year of Graduation (Level 100 → 400)")
ax.set_xlabel("Academic Level")
ax.set_ylabel("Mean GPA")
ax.legend(title="YoG", fontsize=10)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))
plt.tight_layout()
plt.show()

In [ ]:
# ── 4.4 Distribution boxplot across levels ────────────────────────────────────
df_long_large = df_long[df_long["Prog Code"].isin(large_progs)].copy()
df_long_large["Level"] = pd.Categorical(
    df_long_large["Level"], categories=LEVEL_LABELS, ordered=True
)

fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(
    data=df_long_large, x="Level", y="GPA",
    palette="Blues", ax=ax
)
ax.set_title("GPA Distribution Across Academic Levels")
ax.set_xlabel("Academic Level")
ax.set_ylabel("GPA")
plt.tight_layout()
plt.show()

In [ ]:
# ── 4.5 Programme heatmap ─────────────────────────────────────────────────────
pivot = (
    df_long[df_long["Prog Code"].isin(large_progs)]
    .groupby(["Prog Code", "Level"])["GPA"]
    .mean()
    .unstack("Level")
    .round(3)
)
pivot = pivot[LEVEL_LABELS]  # ensure column order
pivot = pivot.sort_values("Level 400", ascending=False)  # sort by final-year GPA

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    pivot, annot=True, fmt=".2f",
    cmap="YlGnBu", vmin=2.5, vmax=4.5,
    linewidths=0.5, ax=ax
)
ax.set_title("Mean GPA per Programme × Level (sorted by Level 400)")
ax.set_xlabel("Academic Level")
ax.set_ylabel("Programme")
plt.tight_layout()
plt.show()

### 4.6 — Per-student trajectory (enriched columns)

`Trajectory_Slope` and `Trajectory_Class` are computed in `enrich.py` (OLS slope across four level GPAs; Improving / Stable / Declining at ±0.05).

In [ ]:
# ── 4.6 Trajectory slope distribution ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["Trajectory_Slope"], bins=40, kde=True, ax=axes[0], color="steelblue")
axes[0].set_title("Distribution of Trajectory_Slope")
df["Trajectory_Class"].value_counts().sort_index().plot(
    kind="bar", ax=axes[1], color=["#4CAF50", "#FF9800", "#F44336"]
)
axes[1].set_title("Trajectory_Class counts")
axes[1].set_xlabel("")
plt.tight_layout()
plt.show()

print("Crosstab: Trajectory_Class × Gender")
print(pd.crosstab(df["Trajectory_Class"], df["Gender"]))
print("\nCrosstab: Trajectory_Class × Prog Code (top 8 programmes)")
top8 = df["Prog Code"].value_counts().head(8).index
print(pd.crosstab(df.loc[df["Prog Code"].isin(top8), "Trajectory_Class"],
                  df.loc[df["Prog Code"].isin(top8), "Prog Code"]))

### 4.7 — AR(1) panel model (GPA_t ~ GPA_lag)

Each student contributes three level transitions (100→200, 200→300, 300→400). This estimates academic **momentum** independent of synthesized predictors.

In [ ]:
# ── 4.7 AR(1) on stacked panel ────────────────────────────────────────────────
panel_rows = []
for _, row in df.iterrows():
    gpas = [row[c] for c in LEVEL_COLS]
    for i in range(1, 4):
        panel_rows.append({"GPA_t": gpas[i], "GPA_lag": gpas[i - 1]})
panel = pd.DataFrame(panel_rows)

ar1 = sm.OLS(panel["GPA_t"], sm.add_constant(panel["GPA_lag"])).fit()
print(ar1.summary())

### 4.8 — Cohort mean trend (Kendall τ)

Non-parametric trend test on the four-point cohort mean GPA series (levels 1–4).

In [ ]:
# ── 4.8 Kendall τ on cohort mean trajectory ───────────────────────────────────
from scipy.stats import kendalltau

mean_series = df[LEVEL_COLS].mean()
tau, p_tau = kendalltau([1, 2, 3, 4], mean_series.values)
print("Cohort mean GPA by level:")
print(mean_series.round(3).to_string())
print(f"\nKendall τ = {tau:.4f},  p = {p_tau:.4f}")

---
## Section 5 — Limitations

**This section is a required disclosure. The text below must appear in any paper, report, or
presentation that uses this dataset or the regression results derived from it.**

---

### L1 — Residual Endogeneity *(most critical)*

`Attendance_Rate` and `Study_Hours_Per_Week` are synthesized using `CGPA` as an input (at 40%
weight via the blended generative model). Running OLS regression of `CGPA` on these variables
therefore recovers a relationship that is **partly constructed**. The significant p-values and
positive coefficients do **not** constitute evidence that higher attendance or more study hours
cause better academic outcomes. They demonstrate that the regression machinery functions correctly
on a dataset where such a relationship was imposed.

The paper must state this explicitly and prominently in the methodology section — not just in a
limitations appendix.

---

### L2 — `Previous_GPA` and dependent variable

Primary OLS uses **`CGPA400`**; `Previous_GPA = mean(CGPA100–300)` is arithmetically prior (no shared
level component). Adj R² ≈ 0.65 is lower than models on overall `CGPA` (≈ 0.95), where three of four
CGPA components overlap `Previous_GPA`. Legacy overall-CGPA metrics remain available for comparison.

---

### L3 — Synthesized Variables Are Not Observed Data

`Attendance_Rate`, `Study_Hours_Per_Week`, and `Course_Load` were never measured. They are
statistically engineered. Any coefficient estimates for these variables carry **no empirical
weight** and should not be used to make policy recommendations.

---

### L4 — Overload Rate Assumption

The 5% overload probability and the 16–20 course overload ceiling are assumptions calibrated
against a single verified MTU transcript. If broader institutional data is available,
`OVERLOAD_PROB`, `OVERLOAD_LO`, and `OVERLOAD_HI` in `enrich.py` should be updated accordingly.

---

### L5 — ID Number Collisions in Source Data

70 student ID numbers appear on rows with conflicting attributes (different programme, gender, or
graduation year). These are presumed to be data-entry errors. This notebook drops duplicate ID
rows (keeping first) before analysis. Any use of `ID No` as a join key should account for this.

---

### L6 — Generalizability

The synthesis parameters were calibrated for a Nigerian university context (Mountain Top
University). The population means, standard deviations, and bounds for `Attendance_Rate`,
`Study_Hours_Per_Week`, and `Course_Load` are not directly transferable to other institutions.